# Trabajo Práctico Integrador: Hitos 1 y 2
**Integrantes del grupo:**
1. Nicolas Llaneza
2. David Lopez
3. Sabrina Moreira
4. Tomas Ferro
5. Emmanuel Avellaneda

---

### Dataset Elegido

**Nombre:** `Students Performance Dataset.csv`  
**Fuente:** Kaggle

---

### Preguntas de Negocio / Pedagógicas

#### Pregunta 1 — Predicción de Riesgo de Abandono
> **¿Qué combinación de variables conductuales y académicas (asistencia, nivel de estrés, horas de estudio y nota del parcial) permite identificar, con mayor precisión, a los estudiantes con alta probabilidad de obtener una calificación final reprobatoria (F) antes de que finalice el cursado?**

**Hipótesis:** Estudiantes con asistencia inferior al 65 %, nota de parcial inferior a 50 y nivel de estrés superior a 7 concentran la mayor parte de las reprobaciones finales, independientemente del departamento académico.

#### Pregunta 2 — Eficiencia del Esfuerzo Académico
> **¿Existe una relación no lineal entre las horas semanales de estudio y el rendimiento académico final, y en qué punto el incremento de horas deja de producir mejoras significativas en la nota total (rendimiento marginal decreciente)?**

**Hipótesis:** A partir de las 20 horas semanales el incremento en `Total_Score` se estabiliza, y los estudiantes con alto estrés no obtienen beneficio adicional aunque aumenten sus horas de estudio.

#### Pregunta 3 — Brecha de Equidad por Contexto Socioeconómico
> **¿En qué medida el nivel de ingresos familiar y el acceso a Internet en el hogar condicionan el desempeño académico, y qué departamentos presentan mayor dispersión de resultados asociada a esas variables de contexto?**

**Hipótesis:** Estudiantes de bajos ingresos sin acceso a Internet en el hogar muestran un promedio de `Total_Score` significativamente inferior al grupo de altos ingresos con acceso, siendo la brecha más pronunciada en departamentos con alta carga de trabajo autónomo

---

### Paso 0 — Importación de Librerías

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)
print('Librerías importadas correctamente')

Librerías importadas correctamente


### Paso 1 — Carga del Dataset


In [38]:
url = "https://raw.githubusercontent.com/Sabrinavmr/TPI-Analisis-Datos/refs/heads/main/Students_Grading_Dataset_Biased.csv"

df = pd.read_csv(url)

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (5000, 23)


,Student_ID,First_Name,Last_Name,Email,Gender,Age,Department,Attendance (%),Midterm_Score,Final_Score,Assignments_Avg,Quizzes_Avg,Participation_Score,Projects_Score,Total_Score,Grade,Study_Hours_per_Week,Extracurricular_Activities,Internet_Access_at_Home,Parent_Education_Level,Family_Income_Level,Stress_Level (1-10),Sleep_Hours_per_Night
0,S1000,Omar,Williams,student0@university.com,Female,22,Engineering,52.29,55.03,57.82,84.22,74.06,3.99,85.90,56.09,F,6.2,No,Yes,High School,Medium,5,4.7
1,S1001,Maria,Brown,student1@university.com,Male,18,Engineering,97.27,97.23,45.80,NaN,94.24,8.32,55.65,50.64,A,19.0,No,Yes,NaN,Medium,4,9.0
2,S1002,Ahmed,Jones,student2@university.com,Male,24,Business,57.19,67.05,93.68,67.70,85.70,5.05,73.79,70.30,D,20.7,No,Yes,Master's,Low,6,6.2
3,S1003,Omar,Williams,student3@university.com,Female,24,Mathematics,95.15,47.79,80.63,66.06,93.51,6.54,92.12,61.63,A,24.8,Yes,Yes,High School,High,3,6.7
4,S1004,John,Smith,student4@university.com,Female,23,CS,54.18,46.59,78.89,96.85,83.70,5.97,68.42,66.13,F,15.4,Yes,Yes,High School,High,2,7.1


In [39]:
print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

if df.shape[0] >= 5000:
    print("El dataset cumple con el requisito mínimo de 5000 registros.")
else:
    print("El dataset NO cumple con el requisito mínimo requerido.")

Cantidad de filas: 5000
Cantidad de columnas: 23
El dataset cumple con el requisito mínimo de 5000 registros.


### Paso 2 — Auditoría Inicial del Dataset

In [40]:
print('── Tipos de datos ──────────────────────────')
print(df.dtypes)
print()
print('── Estadísticas descriptivas ───────────────')
df.describe().round(2)

── Tipos de datos ──────────────────────────
Student_ID                     object
First_Name                     object
Last_Name                      object
Email                          object
Gender                         object
Age                             int64
Department                     object
Attendance (%)                float64
Midterm_Score                 float64
Final_Score                   float64
Assignments_Avg               float64
Quizzes_Avg                   float64
Participation_Score           float64
Projects_Score                float64
Total_Score                   float64
Grade                          object
Study_Hours_per_Week          float64
Extracurricular_Activities     object
Internet_Access_at_Home        object
Parent_Education_Level         object
Family_Income_Level            object
Stress_Level (1-10)             int64
Sleep_Hours_per_Night         float64
dtype: object

── Estadísticas descriptivas ───────────────


,Age,Attendance (%),Midterm_Score,Final_Score,Assignments_Avg,Quizzes_Avg,Participation_Score,Projects_Score,Total_Score,Study_Hours_per_Week,Stress_Level (1-10),Sleep_Hours_per_Night
count,5000.00,4484.00,5000.00,5000.00,4483.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,21.05,75.43,70.33,69.64,74.80,74.91,4.98,74.92,75.12,17.66,5.48,6.49
std,1.99,14.37,17.21,17.24,14.41,14.50,2.89,14.42,14.40,7.28,2.86,1.45
min,18.00,50.01,40.00,40.00,50.00,50.03,0.00,50.01,50.02,5.00,1.00,4.00
25%,19.00,63.26,55.46,54.67,62.09,62.49,2.44,62.32,62.84,11.40,3.00,5.20
50%,21.00,75.72,70.51,69.74,74.81,74.69,4.96,74.98,75.40,17.50,5.00,6.50
75%,23.00,87.47,84.97,84.50,86.97,87.63,7.50,87.37,87.65,24.10,8.00,7.70
max,24.00,100.00,99.98,99.98,99.98,99.96,10.00,100.00,99.99,30.00,10.00,9.00


In [41]:
print('── Valores nulos por columna ───────────────')
nulos = df.isnull().sum()
display(nulos[nulos > 0].rename('Nulos').to_frame())
print(f'Total de nulos: {nulos.sum()}')

# ── Detección y tratamiento de duplicados
n_dup = df.duplicated().sum()
print(f'\nFilas duplicadas detectadas: {n_dup}')

if n_dup > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Se eliminaron {n_dup} filas duplicadas. Dataset: {len(df):,} registros.')
else:
    print('No se detectaron duplicados. No se requiere acción.')

── Valores nulos por columna ───────────────


,Nulos
Attendance (%),516
Assignments_Avg,517
Parent_Education_Level,1794


Total de nulos: 2827

Filas duplicadas detectadas: 0
No se detectaron duplicados. No se requiere acción.


### Paso 3 — Tratamiento de Valores Nulos

La única columna con valores nulos es `Parent_Education_Level` (≈ 20 % del dataset).  
**Criterio adoptado:** imputación con la categoría `'Unknown'` porque no es válido asumir el nivel educativo de los padres de ningún estudiante, y eliminar esos registros introduciría sesgo en el análisis de equidad.

In [43]:
df['Parent_Education_Level'] = df['Parent_Education_Level'].fillna('Unknown')

assert df.isnull().sum().sum() == 0, '❌ Aún quedan valores nulos en el dataset.'
print(f'Nulos restantes: {df.isnull().sum().sum()}')

AssertionError: ❌ Aún quedan valores nulos en el dataset.

### Paso 4 — Normalización de Strings


In [44]:
cols_categoricas = [
    'Gender', 'Department', 'Extracurricular_Activities',
    'Internet_Access_at_Home', 'Parent_Education_Level',
    'Family_Income_Level', 'Grade'
]

for col in cols_categoricas:
    df[col] = df[col].str.strip().str.title()

print('Strings normalizados (strip + title case)\n')

for col in ['Gender', 'Grade', 'Family_Income_Level', 'Department']:
    print(f'{col:30s}: {sorted(df[col].unique())}')

Strings normalizados (strip + title case)

Gender                        : ['Female', 'Male']
Grade                         : ['A', 'B', 'C', 'D', 'F']
Family_Income_Level           : ['High', 'Low', 'Medium']
Department                    : ['Business', 'Cs', 'Engineering', 'Mathematics']


### Paso 5 — Detección y Eliminación de Outliers (Método IQR)


In [45]:
cols_numericas = [
    'Attendance (%)', 'Midterm_Score', 'Final_Score',
    'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score',
    'Projects_Score', 'Total_Score', 'Study_Hours_per_Week',
    'Sleep_Hours_per_Night'
]

filas_antes = len(df)
mascara_outliers = pd.Series([False] * len(df), index=df.index)

for col in cols_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_col = (df[col] < lower) | (df[col] > upper)
    cantidad = outliers_col.sum()
    if cantidad > 0:
        print(f'  {col:30s}: {cantidad:4d} outliers  [rango válido: {lower:.2f} – {upper:.2f}]')
    mascara_outliers = mascara_outliers | outliers_col

df = df[~mascara_outliers].reset_index(drop=True)

print(f'Filas eliminadas por outliers: {filas_antes - len(df)}')
print(f'   Dataset resultante: {len(df):,} registros')

Filas eliminadas por outliers: 0
   Dataset resultante: 5,000 registros


### Paso 6 — Feature Engineering: Creación de Nuevas Variables


#### 6.1 — Índice de Riesgo Académico (`Indice_Riesgo`)

El índice sintetiza en un único valor (escala 0–100, mayor = más riesgo) cuatro señales de alerta:

In [46]:
# ── Límites reales del dataset limpio
stress_min = df['Stress_Level (1-10)'].min()   # = 1
stress_max = df['Stress_Level (1-10)'].max()   # = 10
hours_max  = df['Study_Hours_per_Week'].max()  # máximo real post-outliers

# ── Componentes normalizados a [0, 100]
inasistencia_norm = 100 - df['Attendance (%)']
parcial_norm      = 100 - df['Midterm_Score']
estres_norm       = (df['Stress_Level (1-10)'] - stress_min) / (stress_max - stress_min) * 100
horas_norm = ((hours_max - df['Study_Hours_per_Week']) / max(hours_max, 1) * 100)


# ── Índice ponderado
df['Indice_Riesgo'] = (
    inasistencia_norm * 0.35 +
    parcial_norm      * 0.30 +
    estres_norm       * 0.20 +
    horas_norm        * 0.15
).round(2)

print(f'Indice_Riesgo creado  →  rango real: [{df["Indice_Riesgo"].min():.1f}, {df["Indice_Riesgo"].max():.1f}]')
print(f'   Media: {df["Indice_Riesgo"].mean():.2f}   |   Desv. estándar: {df["Indice_Riesgo"].std():.2f}')

Indice_Riesgo creado  →  rango real: [3.6, 65.0]
   Media: 33.62   |   Desv. estándar: 10.15


#### 6.2 — Categoría de Riesgo (`Categoria_Riesgo`)

In [47]:
def clasificar_riesgo(valor):
    if valor < 30:
        return 'Bajo'
    elif valor < 55:
        return 'Medio'
    else:
        return 'Alto'

df['Categoria_Riesgo'] = df['Indice_Riesgo'].apply(clasificar_riesgo)

print('Categoria_Riesgo creada')
print(df['Categoria_Riesgo'].value_counts().to_string())

Categoria_Riesgo creada
Categoria_Riesgo
Medio    2792
Bajo     1627
Alto      581


#### 6.3 — Delta Parcial vs Final (`Delta_Parcial_Final`)

Mide la variación entre la nota del parcial y la nota final. Valores negativos indican caída en el rendimiento.

In [48]:
df['Delta_Parcial_Final'] = (df['Final_Score'] - df['Midterm_Score']).round(2)
print('Delta_Parcial_Final creada')
print(df['Delta_Parcial_Final'].describe().round(2))

Delta_Parcial_Final creada
count    5000.00
mean       -0.69
std        24.35
min       -58.64
25%       -17.98
50%        -0.67
75%        17.03
max        58.64
Name: Delta_Parcial_Final, dtype: float64


#### 6.4 — Promedio de Evaluaciones Continuas (`Promedio_Continuo`)

Promedio de las cuatro evaluaciones formativas: trabajos prácticos, quizzes, participación y proyectos.

In [49]:
df['Promedio_Continuo'] = (
    (df['Assignments_Avg'] + df['Quizzes_Avg'] +
     df['Participation_Score'] + df['Projects_Score']) / 4
).round(2)
print('Promedio_Continuo creada')

Promedio_Continuo creada


#### 6.5 — Eficiencia de Estudio (`Eficiencia_Estudio`)

Relación entre puntaje total obtenido y horas semanales de estudio. Permite identificar estudiantes que rinden mucho con poco tiempo vs. los que estudian mucho y rinden poco.

In [50]:
df['Eficiencia_Estudio'] = (
    df['Total_Score'] / df['Study_Hours_per_Week'].replace(0, np.nan)
).round(2)

print('Eficiencia_Estudio creada')

#documentar los registros afectados
nulos_eficiencia = df['Eficiencia_Estudio'].isna().sum()
if nulos_eficiencia > 0:
    print(f'estudiantes con 0 horas de estudio (Eficiencia_Estudio = NaN): {nulos_eficiencia}')
else:
    print('Eficiencia_Estudio creada sin valores nulos (ningún estudiante registró 0 horas)')


Eficiencia_Estudio creada
Eficiencia_Estudio creada sin valores nulos (ningún estudiante registró 0 horas)


### Paso 7 — Resumen Final del Dataset Limpio

In [51]:
print('=' * 52)
print('  RESUMEN DEL DATASET LIMPIO')
print('=' * 52)
print(f'  Registros finales  : {df.shape[0]:,}')
print(f'  Variables totales  : {df.shape[1]}')
print(f'  Nulos restantes    : {df.isnull().sum().sum()}')
print()
print('  Distribución de Riesgo:')
for cat, cnt in df['Categoria_Riesgo'].value_counts().items():
    print(f'    {cat:8s}: {cnt:5d} ({cnt/len(df)*100:.1f}%)')
print()
print('  Distribución de Grades:')
print(df['Grade'].value_counts().sort_index().to_string())

df.head()

  RESUMEN DEL DATASET LIMPIO
  Registros finales  : 5,000
  Variables totales  : 28
  Nulos restantes    : 2066

  Distribución de Riesgo:
    Medio   :  2792 (55.8%)
    Bajo    :  1627 (32.5%)
    Alto    :   581 (11.6%)

  Distribución de Grades:
Grade
A    1495
B     978
C     794
D     889
F     844


,Student_ID,First_Name,Last_Name,Email,Gender,Age,Department,Attendance (%),Midterm_Score,Final_Score,Assignments_Avg,Quizzes_Avg,Participation_Score,Projects_Score,Total_Score,Grade,Study_Hours_per_Week,Extracurricular_Activities,Internet_Access_at_Home,Parent_Education_Level,Family_Income_Level,Stress_Level (1-10),Sleep_Hours_per_Night,Indice_Riesgo,Categoria_Riesgo,Delta_Parcial_Final,Promedio_Continuo,Eficiencia_Estudio
0,S1000,Omar,Williams,student0@university.com,Female,22,Engineering,52.29,55.03,57.82,84.22,74.06,3.99,85.90,56.09,F,6.2,No,Yes,High School,Medium,5,4.7,50.98,Medio,2.79,62.04,9.05
1,S1001,Maria,Brown,student1@university.com,Male,18,Engineering,97.27,97.23,45.80,NaN,94.24,8.32,55.65,50.64,A,19.0,No,Yes,Unknown,Medium,4,9.0,13.95,Bajo,-51.43,NaN,2.67
2,S1002,Ahmed,Jones,student2@university.com,Male,24,Business,57.19,67.05,93.68,67.70,85.70,5.05,73.79,70.30,D,20.7,No,Yes,Master'S,Low,6,6.2,40.63,Medio,26.63,58.06,3.40
3,S1003,Omar,Williams,student3@university.com,Female,24,Mathematics,95.15,47.79,80.63,66.06,93.51,6.54,92.12,61.63,A,24.8,Yes,Yes,High School,High,3,6.7,24.40,Bajo,32.84,64.56,2.49
4,S1004,John,Smith,student4@university.com,Female,23,Cs,54.18,46.59,78.89,96.85,83.70,5.97,68.42,66.13,F,15.4,Yes,Yes,High School,High,2,7.1,41.58,Medio,32.30,63.74,4.29


### Paso 8 — Exportación del Dataset Limpio


In [52]:
df.to_csv('students_clean.csv', index=False)
print('Archivo guardado como: students_clean.csv')
print(f'   Dimensiones exportadas: {df.shape[0]:,} filas × {df.shape[1]} columnas')

# Descargar en Google Colab
try:
    from google.colab import files
    files.download('students_clean.csv')
except ImportError:
    print('   (Entorno local detectado — el archivo ya fue guardado en el directorio actual)')

Archivo guardado como: students_clean.csv
   Dimensiones exportadas: 5,000 filas × 28 columnas


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>